In [1]:
# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Data Preprocessing

In [2]:
import pandas as pd
import json

project_dir = '/content/drive/MyDrive/VisualReview'

print("Loading reviews...")
reviews = []
with open(f'{project_dir}/Appliances.jsonl/Appliances.jsonl') as f:
    for line in f:
        reviews.append(json.loads(line))
reviews_df = pd.DataFrame(reviews)
print(f"Reviews loaded: {len(reviews_df):,}")

print("Loading metadata...")
meta = []
with open(f'{project_dir}/meta_Appliances.jsonl/meta_Appliances.jsonl') as f:
    for line in f:
        meta.append(json.loads(line))
meta_df = pd.DataFrame(meta)
print(f"Meta loaded: {len(meta_df):,}")

Loading reviews...
Reviews loaded: 2,128,605
Loading metadata...
Meta loaded: 94,327


In [3]:
# Cell 2: Inspect both dataframes
print("=== REVIEWS COLUMNS ===")
print(reviews_df.columns.tolist())
print("\n=== REVIEWS SAMPLE ===")
print(reviews_df.head(2))

print("\n=== META COLUMNS ===")
print(meta_df.columns.tolist())
print("\n=== META SAMPLE ===")
print(meta_df.head(2))

=== REVIEWS COLUMNS ===
['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']

=== REVIEWS SAMPLE ===
   rating              title                                   text images  \
0     5.0         Work great  work great. use a new one every month     []   
1     5.0  excellent product                Little on the thin side     []   

         asin parent_asin                       user_id      timestamp  \
0  B01N0TQ0OH  B01N0TQ0OH  AGKHLEW2SOWHNMFQIJGBECAF7INQ  1519317108692   
1  B07DD2DMXB  B07DD37QPZ  AHWWLSPCJMALVHDDVSUGICL6RUCA  1664746863446   

   helpful_vote  verified_purchase  
0             0               True  
1             0               True  

=== META COLUMNS ===
['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together', 'subtitle', 'author']

=== META SAMPLE ===
          

In [4]:
# Cell 3 (FIXED): Keep only columns we need
reviews_clean = reviews_df[[
    'parent_asin',
    'rating',
    'text',
]].copy()

meta_clean = meta_df[[
    'parent_asin',
    'title',
    'images',
    'main_category',
]].copy()

print("Reviews shape:", reviews_clean.shape)
print("Meta shape:", meta_clean.shape)

Reviews shape: (2128605, 3)
Meta shape: (94327, 4)


In [5]:
# Cell 4 (FIXED): Extract image URL — images is a LIST of dicts
def extract_image_url(images):
    try:
        if not images or len(images) == 0:
            return None
        # each item is a dict with thumb, large, hi_res, variant
        # find the MAIN image first, fall back to first available
        for img in images:
            if img.get('variant') == 'MAIN':
                return img.get('hi_res') or img.get('large') or None
        # fallback: just return first image's hi_res or large
        first = images[0]
        return first.get('hi_res') or first.get('large') or None
    except:
        return None

meta_clean = meta_clean.copy()
meta_clean['image_url'] = meta_clean['images'].apply(extract_image_url)

print("Products with valid image URL:", meta_clean['image_url'].notna().sum())
print("Products without image URL:", meta_clean['image_url'].isna().sum())
print("\nSample image URL:", meta_clean['image_url'].dropna().iloc[0])

Products with valid image URL: 94326
Products without image URL: 1

Sample image URL: https://m.media-amazon.com/images/I/61zNIJh6ZCL._SL1500_.jpg


In [6]:
# Cell 5 (FIXED): Join reviews with metadata
merged_df = reviews_clean.merge(
    meta_clean[['parent_asin', 'image_url', 'title']],
    on='parent_asin',
    how='inner'
)

print("Merged shape:", merged_df.shape)
print("\nSample row:")
print(merged_df.iloc[0])

Merged shape: (2128605, 5)

Sample row:
parent_asin                                           B01N0TQ0OH
rating                                                       5.0
text                       work great. use a new one every month
image_url      https://m.media-amazon.com/images/I/71zqD75W03...
title          Geesta 12-Pack Premium Activated Charcoal Wate...
Name: 0, dtype: object


In [7]:
# Cell 6 (FIXED): Filter low quality records
print("Before filtering:", f"{len(merged_df):,}")

filtered_df = merged_df[
    merged_df['image_url'].notna() &
    merged_df['text'].notna() &
    (merged_df['text'].str.len() >= 50) &
    (merged_df['text'].str.len() <= 1000) &
    merged_df['rating'].notna()
].copy()

print("After filtering:", f"{len(filtered_df):,}")
print("\nRating distribution:")
print(filtered_df['rating'].value_counts().sort_index())

Before filtering: 2,128,605
After filtering: 1,391,466

Rating distribution:
rating
1.0    201596
2.0     66238
3.0     83407
4.0    144849
5.0    895376
Name: count, dtype: int64


In [8]:
# Cell 7: Keep products with at least 3 reviews
reviews_per_product = filtered_df.groupby('parent_asin')['rating'].agg(
    count='count',
    unique_ratings=lambda x: x.nunique()
).reset_index()

good_products = reviews_per_product[reviews_per_product['count'] >= 3]['parent_asin']
filtered_df = filtered_df[filtered_df['parent_asin'].isin(good_products)].copy()

print(f"Products with 3+ reviews: {len(good_products):,}")
print(f"Total records remaining: {len(filtered_df):,}")

Products with 3+ reviews: 38,927
Total records remaining: 1,340,320


In [9]:
# Cell 8: Save
import os
os.makedirs(f'{project_dir}/processed', exist_ok=True)

output_path = f'{project_dir}/processed/Appliances_joined.jsonl'
filtered_df.to_json(output_path, orient='records', lines=True)
print(f"Saved to {output_path}")
print(f"Final dataset: {len(filtered_df):,} review-image pairs")

Saved to /content/drive/MyDrive/VisualReview/processed/Appliances_joined.jsonl
Final dataset: 1,340,320 review-image pairs
